# A2. Multilingual Listing Batch Generation Notebook

> **Companion module**: [A2 Listing Optimization](../paths/a-operators/a2-listing-optimization.md)
>
> **Function**: Input an English listing, generate 5 language versions in one click (German/Japanese/Spanish/Portuguese/French)
>
> [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kangise/ecommerce-ai-roadmap/blob/main/notebooks/a2-multilingual-listing.ipynb)

---

## 1. Install Dependencies

In [ ]:
!pip install -q openai pandas

## 2. Configure API Key

In [ ]:
import os
from openai import OpenAI
import pandas as pd
import json

# In Colab use Secrets: from google.colab import userdata; key = userdata.get('OPENAI_API_KEY')
OPENAI_API_KEY = os.getenv('OPENAI_API_KEY', 'your-key-here')
client = OpenAI(api_key=OPENAI_API_KEY)
print('API configured')

## 3. Input Your English Listing

In [ ]:
# === Replace with your actual listing ===
ENGLISH_LISTING = {
    'title': 'Premium Wireless Bluetooth Earbuds - Active Noise Cancellation, 36H Battery Life, IPX5 Waterproof, Deep Bass, Touch Control - for iPhone Android',
    'bullets': [
        'ACTIVE NOISE CANCELLATION: Block out background noise with advanced ANC technology. Perfect for commuting, working, and traveling.',
        '36-HOUR BATTERY LIFE: 8 hours per charge + 28 hours with charging case. Quick charge: 10 min = 2 hours playback.',
        'IPX5 WATERPROOF: Sweat and splash resistant. Ideal for gym workouts and outdoor activities.',
        'PREMIUM SOUND: 13mm dynamic drivers deliver deep bass and crystal-clear highs. Bluetooth 5.3 for stable connection.',
        'COMFORTABLE FIT: 4 sizes of silicone ear tips (XS/S/M/L). Lightweight design at only 5g per earbud.'
    ],
    'description': 'Experience premium audio with our wireless earbuds featuring active noise cancellation, 36-hour battery life, and IPX5 waterproof rating. Perfect for music lovers, commuters, and fitness enthusiasts.',
    'backend_keywords': 'wireless earbuds bluetooth noise cancelling waterproof gym workout running'
}

TARGET_LANGUAGES = {
    'de': {'name': 'German', 'market': 'Amazon.de', 'notes': 'Use Sie (formal), emphasize technical specs and certifications'},
    'ja': {'name': 'Japanese', 'market': 'Amazon.co.jp', 'notes': 'Use です/ます体, emphasize 品質/安心/保証, detailed specs'},
    'es': {'name': 'Latin American Spanish', 'market': 'Mercado Libre MX', 'notes': 'Use ustedes (not vosotros), ≤60 char title, mention envío gratis'},
    'pt-br': {'name': 'Brazilian Portuguese', 'market': 'Mercado Libre BR', 'notes': 'Use você (not tu), ≤60 char title, mention frete grátis/parcelamento'},
    'fr': {'name': 'French', 'market': 'Amazon.fr', 'notes': 'Emphasize eco-friendly aspects, CE certification'}
}

print(f'Source: English Listing')
print(f'Target languages: {len(TARGET_LANGUAGES)}')
for code, info in TARGET_LANGUAGES.items():
    print(f'  {code}: {info["name"]} ({info["market"]})')

## 4. AI Batch Translation + Localization

In [ ]:
def localize_listing(listing, lang_code, lang_info):
    """Use AI to localize an English listing to the target language"""
    prompt = f"""You are an expert e-commerce listing localizer for {lang_info['market']}.

Source English Listing:
Title: {listing['title']}
Bullet Points:
{chr(10).join(f'- {b}' for b in listing['bullets'])}
Description: {listing['description']}
Backend Keywords: {listing['backend_keywords']}

Please localize to {lang_info['name']} for {lang_info['market']}.
IMPORTANT: {lang_info['notes']}

This is NOT translation — it's cultural adaptation. Use local search keywords, local expressions, and local consumer expectations.

Return JSON format:
{{
  "title": "localized title",
  "bullets": ["bullet1", "bullet2", "bullet3", "bullet4", "bullet5"],
  "description": "localized description",
  "keywords": ["keyword1", "keyword2", ..., "keyword10"],
  "notes": "localization notes for reviewer"
}}"""
    
    response = client.chat.completions.create(
        model='gpt-4o-mini',
        messages=[{'role': 'user', 'content': prompt}],
        response_format={'type': 'json_object'},
        max_tokens=1500
    )
    return json.loads(response.choices[0].message.content)

# Run for all languages
results = {}
for code, info in TARGET_LANGUAGES.items():
    print(f'\nLocalizing to {info["name"]}...')
    try:
        result = localize_listing(ENGLISH_LISTING, code, info)
        results[code] = result
        print(f'  ✅ Title: {result["title"][:60]}...')
        print(f'  Keywords: {", ".join(result.get("keywords", [])[:3])}...')
    except Exception as e:
        print(f'  ❌ Error: {e}')
        results[code] = {'error': str(e)}

print(f'\n=== Done: {sum(1 for r in results.values() if "error" not in r)}/{len(TARGET_LANGUAGES)} languages ===')


## 5. Results Display

In [ ]:
for code, result in results.items():
    if 'error' in result:
        continue
    info = TARGET_LANGUAGES[code]
    print(f'\n{"="*60}')
    print(f'🌐 {info["name"]} ({info["market"]})')
    print(f'{"="*60}')
    print(f'\nTitle: {result["title"]}')
    print(f'\nBullet Points:')
    for i, b in enumerate(result.get('bullets', []), 1):
        print(f'  {i}. {b}')
    print(f'\nKeywords: {", ".join(result.get("keywords", []))}')
    if result.get('notes'):
        print(f'\n📝 Notes: {result["notes"]}')

## 6. Export to CSV

In [ ]:
rows = []
for code, result in results.items():
    if 'error' in result:
        continue
    rows.append({
        'language': code,
        'market': TARGET_LANGUAGES[code]['market'],
        'title': result.get('title', ''),
        'bullet_1': result.get('bullets', [''])[0] if result.get('bullets') else '',
        'bullet_2': result.get('bullets', ['', ''])[1] if len(result.get('bullets', [])) > 1 else '',
        'bullet_3': result.get('bullets', ['', '', ''])[2] if len(result.get('bullets', [])) > 2 else '',
        'bullet_4': result.get('bullets', ['', '', '', ''])[3] if len(result.get('bullets', [])) > 3 else '',
        'bullet_5': result.get('bullets', ['', '', '', '', ''])[4] if len(result.get('bullets', [])) > 4 else '',
        'description': result.get('description', ''),
        'keywords': ', '.join(result.get('keywords', []))
    })

df = pd.DataFrame(rows)
df.to_csv('multilingual_listings.csv', index=False)
print(f'Exported {len(df)} listings to multilingual_listings.csv')
df